# Resultats — GAN Discriminator (BERT)

Ce notebook presente les resultats du GAN Discriminator a travers les differentes iterations de training.

1. **Comparaison des runs MLflow** — parametres, metriques, temps de training
2. **Evolution entre les versions** — ce qui a change d'une iteration a l'autre
3. **Tests end-to-end avec GAN v4** — pipeline complet + matrice de confusion

## 1. Setup & chargement des donnees MLflow

In [ ]:
import dagshub
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.facecolor'] = 'white'

dagshub.init('NLP-Fact-checking', 'MarcoSrhl', mlflow=True)
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name('fact-checker')
print(f'Experiment: {exp.name} (id={exp.experiment_id})')

In [ ]:
# Recuperer les 4 runs originaux (pas les packages)
all_runs = client.search_runs(
    exp.experiment_id,
    filter_string="params.model_type = 'bert-gan-swap'",
    max_results=50,
    order_by=['start_time ASC'],
)

original_runs = [
    r for r in all_runs
    if not (r.info.run_name or '').startswith('package_')
    and 'source_run_id' not in r.data.params
]

print(f'{len(original_runs)} runs de training trouves')
for i, r in enumerate(original_runs, 1):
    print(f'  v{i}: {r.info.run_name} ({r.info.run_id[:8]})')

## 2. Comparaison des iterations

In [ ]:
# Construire le tableau comparatif
rows = []
for i, r in enumerate(original_runs, 1):
    duration_s = (r.info.end_time - r.info.start_time) / 1000 if r.info.end_time else 0
    m = r.data.metrics
    p = r.data.params
    rows.append({
        'Version': f'v{i}',
        'Run ID': r.info.run_id[:8],
        'Epochs': int(p.get('epochs', 0)),
        'Batch Size': int(p.get('batch_size', 0)),
        'Learning Rate': float(p.get('lr_d', 0)),
        'Generator': p.get('generator_mode', 'random'),
        'Training Time': f'{duration_s:.0f}s ({duration_s/60:.1f} min)',
        'D Loss': m.get('d_loss', None),
        'D Real Score': m.get('d_real_score', None),
        'D Fake Score': m.get('d_fake_score', None),
        'Val Loss': m.get('val_loss', None),
        'Val Accuracy': m.get('val_accuracy', None),
    })

df = pd.DataFrame(rows).set_index('Version')
df

In [ ]:
# Parametres communs vs. differents
print('=== Parametres fixes (toutes les versions) ===')
print(f'  Modele:           BERT (bert-base-uncased)')
print(f'  Architecture:     bert-gan-swap (entity-swap GAN)')
print(f'  Learning Rate:    2e-5 (AdamW)')
print(f'  BERT Layers geles: 10/12')
print(f'  Dropout:          0.4')
print(f'  Label Smoothing:  0.9')
print(f'  Early Stopping:   patience=3, min_delta=0.001')
print(f'  LR Scheduler:     cosine decay avec warmup (10%)')
print(f'  Donnees:          41 categories DBpedia x 5000 triplets')
print(f'  Parametres:       ~109M total (~23M trainables)')
print()
print('=== Parametres variables entre versions ===')
for _, row in df.iterrows():
    print(f"  {row.name}: epochs={row['Epochs']}, batch_size={row['Batch Size']}, "
          f"generator={row['Generator']}, time={row['Training Time']}")

## 3. Visualisation des metriques

In [ ]:
versions = df.index.tolist()
x = np.arange(len(versions))
width = 0.55

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- D Loss ---
ax = axes[0]
vals = df['D Loss'].values
colors = ['#F0AD4E' if v == min(vals) else '#ADB5BD' for v in vals]
ax.bar(x, vals, width, color=colors, edgecolor='white')
ax.set_title('Discriminator Loss', fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(versions)
ax.set_ylabel('Loss')
for i, v in enumerate(vals):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)

# --- Val Accuracy ---
ax = axes[1]
val_acc = df['Val Accuracy'].values
bar_colors = []
valid_vals = [v for v in val_acc if v is not None and not np.isnan(v)]
best_val = max(valid_vals) if valid_vals else None
for v in val_acc:
    if v is None or (isinstance(v, float) and np.isnan(v)):
        bar_colors.append('#E0E0E0')
    elif v == best_val:
        bar_colors.append('#5CB85C')
    else:
        bar_colors.append('#ADB5BD')
display_vals = [v if v is not None and not (isinstance(v, float) and np.isnan(v)) else 0 for v in val_acc]
ax.bar(x, display_vals, width, color=bar_colors, edgecolor='white')
ax.set_title('Validation Accuracy', fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(versions)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.0)
for i, v in enumerate(val_acc):
    if v is not None and not (isinstance(v, float) and np.isnan(v)):
        ax.text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=10)
    else:
        ax.text(i, 0.05, 'N/A', ha='center', fontsize=10, color='gray')

# --- Real vs Fake Scores ---
ax = axes[2]
w2 = 0.25
ax.bar(x - w2/2, df['D Real Score'].values, w2, label='Real Score', color='#5CB85C', edgecolor='white')
ax.bar(x + w2/2, df['D Fake Score'].values, w2, label='Fake Score', color='#D9534F', edgecolor='white')
ax.set_title('Real vs Fake Scores', fontweight='bold', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(versions)
ax.set_ylabel('Score (0=fake, 1=real)')
ax.legend()
ax.set_ylim(0, 1.0)

plt.tight_layout()
plt.savefig('resultats_metriques.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: resultats_metriques.png')

In [ ]:
# Temps de training par version
times_s = []
for r in original_runs:
    d = (r.info.end_time - r.info.start_time) / 1000 if r.info.end_time else 0
    times_s.append(d)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(versions, [t / 60 for t in times_s], color='#4A90D9', edgecolor='white', height=0.5)
ax.set_xlabel('Temps (minutes)')
ax.set_title('Temps de training par version', fontweight='bold', fontsize=12)
for i, (bar, t) in enumerate(zip(bars, times_s)):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{t:.0f}s ({t/60:.1f} min)', va='center', fontsize=10)
ax.set_xlim(0, max(times_s)/60 + 1)
plt.tight_layout()
plt.show()

## 4. Evolution entre les versions

| Version | Changements | Impact |
|---------|------------|--------|
| **v1** | Premier training: batch_size=16, 20 epochs | Meilleur D loss (0.49) et separation real/fake (0.84 vs 0.07) mais pas de metriques de validation |
| **v2** | batch_size=64, seulement 5 epochs | Plus rapide mais sous-entraine (val_acc=75.5%) |
| **v3** | batch_size=64, 20 epochs | Meilleure val_accuracy (82.7%) avec bon equilibre |
| **v4** | batch_size=64, 20 epochs, ajustements | val_accuracy=80.4%, meilleure val_loss (0.826) |

## 5. Tests end-to-end avec le GAN (modele local)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))

from triplet_extractor import TripletExtractor
from entity_linker import EntityLinker
from knowledge_query import KnowledgeQuery
from gan_model import FactGAN

extractor = TripletExtractor()
linker = EntityLinker()
kb = KnowledgeQuery()

gan = FactGAN()
gan.load('../models/gan')
gan.discriminator.eval()
print(f'GAN charge ({sum(p.numel() for p in gan.discriminator.parameters()):,} params)')

In [ ]:
def check_claim(claim, gan_high=0.65, gan_low=0.35):
    """Pipeline complet: claim -> verdict."""
    triplets = extractor.extract(claim)
    if not triplets:
        return {'claim': claim, 'verdict': 'NOT ENOUGH INFO', 'gan_score': None,
                'kb_found': False, 'confidence': 0.0, 'triplets': []}

    scores = gan.discriminate_triplets(triplets)
    avg_score = scores.mean().item()

    entity_uris = {}
    for s, _, o in triplets:
        for ent in [s, o]:
            if ent not in entity_uris:
                entity_uris[ent] = linker.link(ent)

    kb_found = False
    for s, p, o in triplets:
        s_uri, o_uri = entity_uris.get(s), entity_uris.get(o)
        if s_uri or o_uri:
            result = kb.verify_triplet(s_uri, o_uri)
            if result['found']:
                kb_found = True
                break

    if avg_score > gan_high:
        verdict = 'SUPPORTED'
    elif avg_score < gan_low:
        verdict = 'REFUTED'
    else:
        verdict = 'NOT ENOUGH INFO'

    confidence = avg_score if verdict == 'SUPPORTED' else (1 - avg_score)
    if kb_found and verdict == 'SUPPORTED':
        confidence = min(0.99, confidence * 1.2)
    elif not kb_found and verdict == 'SUPPORTED':
        confidence *= 0.8

    return {
        'claim': claim, 'triplets': triplets, 'gan_score': avg_score,
        'kb_found': kb_found, 'verdict': verdict, 'confidence': confidence,
    }

In [ ]:
# Jeu de test — 49 claims avec labels attendus
test_claims = [
    # SUPPORTED (20)
    ('Paris is the capital of France', 'SUPPORTED'),
    ('Rome is the capital of Italy', 'SUPPORTED'),
    ('Tokyo is the capital of Japan', 'SUPPORTED'),
    ('Berlin is the capital of Germany', 'SUPPORTED'),
    ('Madrid is the capital of Spain', 'SUPPORTED'),
    ('Albert Einstein developed the theory of relativity', 'SUPPORTED'),
    ('The Eiffel Tower is located in Paris', 'SUPPORTED'),
    ('Barack Obama was born in Hawaii', 'SUPPORTED'),
    ('William Shakespeare wrote Hamlet', 'SUPPORTED'),
    ('Leonardo da Vinci painted the Mona Lisa', 'SUPPORTED'),
    ('The Amazon River is in South America', 'SUPPORTED'),
    ('Marie Curie won the Nobel Prize', 'SUPPORTED'),
    ('London is the capital of the United Kingdom', 'SUPPORTED'),
    ('Lionel Messi plays for Inter Miami', 'SUPPORTED'),
    ('The Nile is in Africa', 'SUPPORTED'),
    ('Mozart was born in Salzburg', 'SUPPORTED'),
    ('Steve Jobs founded Apple', 'SUPPORTED'),
    ('Napoleon was born in Corsica', 'SUPPORTED'),
    ('The Great Wall is in China', 'SUPPORTED'),
    ('Charles Darwin wrote On the Origin of Species', 'SUPPORTED'),
    # REFUTED (20)
    ('London is the capital of France', 'REFUTED'),
    ('Berlin is the capital of Italy', 'REFUTED'),
    ('Napoleon was born in England', 'REFUTED'),
    ('Tokyo is the capital of France', 'REFUTED'),
    ('Einstein was born in China', 'REFUTED'),
    ('Shakespeare wrote War and Peace', 'REFUTED'),
    ('The Eiffel Tower is in London', 'REFUTED'),
    ('Marie Curie was born in Germany', 'REFUTED'),
    ('The Amazon River is in Europe', 'REFUTED'),
    ('Mozart was born in Paris', 'REFUTED'),
    ('Obama was born in Russia', 'REFUTED'),
    ('Rome is the capital of Germany', 'REFUTED'),
    ('Da Vinci painted Starry Night', 'REFUTED'),
    ('Steve Jobs founded Microsoft', 'REFUTED'),
    ('The Nile is in Australia', 'REFUTED'),
    ('Messi plays for Real Madrid', 'REFUTED'),
    ('Darwin wrote the Bible', 'REFUTED'),
    ('The Great Wall is in India', 'REFUTED'),
    ('Madrid is the capital of Portugal', 'REFUTED'),
    ('Hitler was a nice person', 'REFUTED'),
    # NOT ENOUGH INFO (9)
    ('There is a connection between pizza and quantum physics', 'NOT ENOUGH INFO'),
    ('The meaning of life is 42', 'NOT ENOUGH INFO'),
    ('Aliens have visited Earth', 'NOT ENOUGH INFO'),
    ('Coffee is better than tea', 'NOT ENOUGH INFO'),
    ('Time travel will be possible by 2050', 'NOT ENOUGH INFO'),
    ('Dogs are smarter than cats', 'NOT ENOUGH INFO'),
    ('Music makes plants grow faster', 'NOT ENOUGH INFO'),
    ('The universe has a purpose', 'NOT ENOUGH INFO'),
    ('Chocolate cures depression', 'NOT ENOUGH INFO'),
]

print(f'{len(test_claims)} claims de test')
from collections import Counter
dist = Counter(label for _, label in test_claims)
for label, count in sorted(dist.items()):
    print(f'  {label}: {count}')

In [ ]:
import time

results = []
start = time.time()

for i, (claim, expected) in enumerate(test_claims):
    r = check_claim(claim)
    r['expected'] = expected
    results.append(r)
    match = 'OK' if r['verdict'] == expected else 'MISS'
    score_str = f"{r['gan_score']:.3f}" if r['gan_score'] is not None else 'N/A'
    if (i + 1) % 10 == 0:
        elapsed = time.time() - start
        acc = sum(1 for r2 in results if r2['verdict'] == r2['expected']) / len(results)
        print(f'  [{i+1}/{len(test_claims)}] acc={acc:.0%} | {elapsed:.1f}s')

total_time = time.time() - start
print(f'\nTermine en {total_time:.1f}s ({len(test_claims)/total_time:.1f} claims/s)')

## 6. Resultats detailles

In [ ]:
# Tableau complet des resultats
print(f'{"Claim":55s} {"Expected":18s} {"Predicted":18s} {"GAN":>6s} {"KB":>4s}  Match')
print('-' * 110)

correct = 0
for r in results:
    match = 'OK' if r['expected'] == r['verdict'] else 'MISS'
    if match == 'OK':
        correct += 1
    score_str = f"{r['gan_score']:.3f}" if r['gan_score'] is not None else '  N/A'
    kb_str = 'Y' if r['kb_found'] else 'N'
    print(f"{r['claim']:55s} {r['expected']:18s} {r['verdict']:18s} {score_str:>6s} {kb_str:>4s}  {match}")

print(f'\nAccuracy globale: {correct}/{len(results)} ({correct/len(results):.0%})')

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

labels = ['SUPPORTED', 'REFUTED', 'NOT ENOUGH INFO']
y_true = [r['expected'] for r in results]
y_pred = [r['verdict'] for r in results]

acc = accuracy_score(y_true, y_pred)
precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, zero_division=0
)

print(f'\n{"Label":<20s} {"Precision":>10s} {"Recall":>10s} {"F1":>10s} {"Support":>10s}')
print('-' * 60)
for i, label in enumerate(labels):
    print(f'{label:<20s} {precision[i]:>10.2%} {recall[i]:>10.2%} {f1[i]:>10.2%} {support[i]:>10d}')
print('-' * 60)
print(f'{"GLOBAL":.<20s} {acc:>10.2%}')

## 7. Matrice de confusion

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues', aspect='auto')

short_labels = ['SUPPORTED', 'REFUTED', 'NEI']
ax.set_xticks(range(len(short_labels)))
ax.set_yticks(range(len(short_labels)))
ax.set_xticklabels(short_labels, fontsize=11)
ax.set_yticklabels(short_labels, fontsize=11)
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax.set_title('Matrice de Confusion — GAN Pipeline', fontsize=14, fontweight='bold')

for i in range(len(labels)):
    for j in range(len(labels)):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=16, fontweight='bold', color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

In [ ]:
# Distribution des scores GAN par label
fig, ax = plt.subplots(figsize=(10, 5))

colors_map = {'SUPPORTED': '#5CB85C', 'REFUTED': '#D9534F', 'NOT ENOUGH INFO': '#F0AD4E'}

for label in labels:
    scores = [r['gan_score'] for r in results if r['expected'] == label and r['gan_score'] is not None]
    if scores:
        ax.scatter(
            scores, [label] * len(scores),
            c=colors_map[label], s=80, alpha=0.7, edgecolors='white', linewidth=0.5,
            label=f'{label} (n={len(scores)}, avg={np.mean(scores):.3f})',
        )
        ax.axvline(np.mean(scores), color=colors_map[label], linestyle='--', alpha=0.4)

ax.axvline(0.65, color='green', linestyle=':', alpha=0.6, label='Seuil SUPPORTED (0.65)')
ax.axvline(0.35, color='red', linestyle=':', alpha=0.6, label='Seuil REFUTED (0.35)')
ax.set_xlabel('GAN Score (0=fake, 1=real)', fontsize=11)
ax.set_title('Distribution des scores GAN par label attendu', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: score_distribution.png')

## 8. Analyse des erreurs

In [ ]:
# Claims mal classifies
errors = [r for r in results if r['verdict'] != r['expected']]
print(f'{len(errors)} erreurs sur {len(results)} claims ({len(errors)/len(results):.0%})\n')

print(f'{"Claim":55s} {"Expected":15s} {"Got":15s} {"GAN":>6s} {"KB":>3s}')
print('-' * 100)
for r in errors:
    score_str = f"{r['gan_score']:.3f}" if r['gan_score'] is not None else 'N/A'
    kb_str = 'Y' if r['kb_found'] else 'N'
    print(f"{r['claim']:55s} {r['expected']:15s} {r['verdict']:15s} {score_str:>6s} {kb_str:>3s}")

# Types d'erreurs
print('\n--- Types d\'erreurs ---')
error_types = Counter((r['expected'], r['verdict']) for r in errors)
for (true, pred), count in error_types.most_common():
    print(f'  {true} -> {pred}: {count}')

## 9. Confidence par verdict

In [ ]:
# Confidence moyenne par verdict predit
print(f'{"Verdict predit":<20s} {"Confidence moy.":>15s} {"Count":>8s}')
print('-' * 45)
for verdict in labels:
    subset = [r for r in results if r['verdict'] == verdict]
    if subset:
        avg_conf = np.mean([r['confidence'] for r in subset])
        print(f'{verdict:<20s} {avg_conf:>15.1%} {len(subset):>8d}')

print()
# Confidence quand correct vs incorrect
correct_confs = [r['confidence'] for r in results if r['verdict'] == r['expected']]
wrong_confs = [r['confidence'] for r in results if r['verdict'] != r['expected']]
print(f'Confidence moyenne (predictions correctes):  {np.mean(correct_confs):.1%}')
print(f'Confidence moyenne (predictions incorrectes): {np.mean(wrong_confs):.1%}' if wrong_confs else 'Aucune erreur!')

## 10. Resume

In [ ]:
print('=' * 60)
print('RESUME DES RESULTATS')
print('=' * 60)
print()
print('--- Iterations de training (MLflow) ---')
for i, r in enumerate(original_runs, 1):
    m = r.data.metrics
    d = (r.info.end_time - r.info.start_time) / 1000 if r.info.end_time else 0
    val_str = f"{m.get('val_accuracy', 0):.1%}" if 'val_accuracy' in m else 'N/A'
    print(f'  v{i}: val_acc={val_str}, d_loss={m.get("d_loss", 0):.3f}, time={d:.0f}s')

print()
print('--- Evaluation end-to-end (pipeline complet) ---')
print(f'  Claims testes: {len(results)}')
print(f'  Accuracy globale: {acc:.0%}')
for i, label in enumerate(labels):
    print(f'  {label}: precision={precision[i]:.0%}, recall={recall[i]:.0%}, f1={f1[i]:.0%}')

print()
print('--- Architecture ---')
print(f'  Modele: BERT (bert-base-uncased) + classification head')
print(f'  Generator: entity-swap (SwapGenerator)')
print(f'  Donnees: 41 categories DBpedia x 5000 triplets')
print(f'  Parametres: ~109M total')